In [55]:
import numpy as np
from pyscf import gto, scf
from pyscf.scf import uhf

# Geometry 1
mol1 = gto.M(
    atom='''
    N 0.000000 0.000000 0.000000
    N 0.000000 0.000000 1.10
    ''',
    basis='cc-pvdz',
    unit='Angstrom',
    verbose=0
)
mf1 = scf.RHF(mol1).run()

# Geometry 2
mol2 = gto.M(
    atom='''
    N 0.000000 0.000000 0.000000
    N 0.000000 0.000000 3.0
    ''',
    basis='cc-pvdz',
    unit='Angstrom',
    verbose=0
)
mf2 = scf.RHF(mol2).run()


In [56]:
from pyscf.tools import mo_mapping

idx, s = mo_mapping.mo_map(mol1, mf1.mo_coeff, mol2, mf2.mo_coeff, base=0, tol=0.3)
print("Pairs with |overlap| > 0.3:")
for i, j in idx:
    print(f"{i:3d}  {j:3d}   s = {s[i,j]: .8f}   |s| = {abs(s[i,j]):.8f}")

Pairs with |overlap| > 0.3:
  0    0   s =  0.49881686   |s| = 0.49881686
  0    1   s =  0.50133316   |s| = 0.50133316
  1    0   s = -0.49864151   |s| = 0.49864151
  1    1   s = -0.50119503   |s| = 0.50119503
  2    2   s =  0.57695554   |s| = 0.57695554
  2    3   s =  0.51404201   |s| = 0.51404201
  2    4   s = -0.39469403   |s| = 0.39469403
  2    9   s =  0.36532675   |s| = 0.36532675
  3    3   s =  0.46888696   |s| = 0.46888696
  3    4   s =  0.55731189   |s| = 0.55731189
  3   10   s = -0.37738235   |s| = 0.37738235
  4    2   s = -0.34341122   |s| = 0.34341122
  4    9   s =  0.53642599   |s| = 0.53642599
  4   10   s = -0.49765466   |s| = 0.49765466
  5    6   s =  0.62117892   |s| = 0.62117892
  5    8   s =  0.54413227   |s| = 0.54413227
  6    5   s =  0.62117892   |s| = 0.62117892
  6    7   s =  0.54413227   |s| = 0.54413227
  7    5   s = -0.30316139   |s| = 0.30316139
  7    7   s = -0.46792164   |s| = 0.46792164
  8    6   s =  0.30316139   |s| = 0.30316139
  8   

In [57]:
import numpy as np
from pyscf import lo

def meta_lowdin_ao_weights_occ(mf):
    mol = mf.mol
    S = mf.get_ovlp()
    X = lo.orth.orth_ao(mol, method='meta_lowdin', s=S)

    occ_idx = np.where(mf.mo_occ > 0)[0]

    C_occ = mf.mo_coeff[:, occ_idx]
    C_oao_occ = X.conj().T @ S @ C_occ

    w_occ = np.sum(np.abs(C_oao_occ)**2, axis=1)
    labels = mol.ao_labels()
    return labels, C_oao_occ, w_occ
def compare_meta_lowdin_occ(mf1, mf2, topn=20, min_weight=0.1):
    labels1, C1, w1 = meta_lowdin_ao_weights_occ(mf1)
    labels2, C2, w2 = meta_lowdin_ao_weights_occ(mf2)

    if labels1 != labels2:
        raise ValueError("AO labels differ between geometries; need a more careful cross-geometry mapping.")

    dw = w2 - w1

    # keep only rows where either geometry has weight > min_weight
    keep = (w1 > min_weight) | (w2 > min_weight)
    idx_keep = np.where(keep)[0]

    # sort only the kept rows by |delta|
    order = idx_keep[np.argsort(np.abs(dw[idx_keep]))[::-1]]

    print(f"Filter: w1 > {min_weight} or w2 > {min_weight}")
    print(f"{'idx':>4}  {'AO label':<30}  {'w1':>12}  {'w2':>12}  {'delta':>12}")
    print("-" * 78)

    for k in order[:topn]:
        print(f"{k:4d}  {labels1[k]:<30}  {w1[k]:12.8f}  {w2[k]:12.8f}  {dw[k]:12.4e}")

    return labels1, w1, w2, dw

In [58]:
def meta_lowdin_ao_weights_vir(mf):
    mol = mf.mol
    S = mf.get_ovlp()
    X = lo.orth.orth_ao(mol, method='meta_lowdin', s=S)

    vir_idx = np.where(mf.mo_occ == 0)[0]

    C_vir = mf.mo_coeff[:, vir_idx]
    C_oao_vir = X.conj().T @ S @ C_vir

    w_vir = np.sum(np.abs(C_oao_vir)**2, axis=1)
    labels = mol.ao_labels()
    return labels, C_oao_vir, w_vir


def compare_meta_lowdin_vir(mf1, mf2, topn=20, min_weight=0.1):
    labels1, C1, w1 = meta_lowdin_ao_weights_vir(mf1)
    labels2, C2, w2 = meta_lowdin_ao_weights_vir(mf2)

    if labels1 != labels2:
        raise ValueError("AO labels differ between geometries; need a more careful cross-geometry mapping.")

    dw = w2 - w1

    keep = (w1 > min_weight) | (w2 > min_weight)
    idx_keep = np.where(keep)[0]
    order = idx_keep[np.argsort(np.abs(dw[idx_keep]))[::-1]]

    print(f"Filter: w1 > {min_weight} or w2 > {min_weight}")
    print(f"{'idx':>4}  {'AO label':<30}  {'w1':>12}  {'w2':>12}  {'delta':>12}")
    print("-" * 78)

    for k in order[:topn]:
        print(f"{k:4d}  {labels1[k]:<30}  {w1[k]:12.8f}  {w2[k]:12.8f}  {dw[k]:12.4e}")

    return labels1, w1, w2, dw

In [70]:
labelsocc, w1occ, w2occ, dwocc = compare_meta_lowdin_occ(mf1, mf2, topn=100)

Filter: w1 > 0.1 or w2 > 0.1
 idx  AO label                                  w1            w2         delta
------------------------------------------------------------------------------
   1  0 N 2s                            0.79647302    0.99819383    2.0172e-01
  15  1 N 2s                            0.79647302    0.99819383    2.0172e-01
   5  0 N 2pz                           0.68569073    0.49391675   -1.9177e-01
  19  1 N 2pz                           0.68569073    0.49391750   -1.9177e-01
   3  0 N 2px                           0.49691282    0.49476482   -2.1480e-03
   4  0 N 2py                           0.49691282    0.49476482   -2.1480e-03
  18  1 N 2py                           0.49691282    0.49476582   -2.1470e-03
  17  1 N 2px                           0.49691282    0.49476582   -2.1470e-03
  14  1 N 1s                            0.99999363    0.99999741    3.7848e-06
   0  0 N 1s                            0.99999363    0.99999741    3.7848e-06


In [71]:
labelsvir, w1vir, w2vir, dwvir = compare_meta_lowdin_vir(mf1, mf2, topn=100, min_weight=0.1)

Filter: w1 > 0.1 or w2 > 0.1
 idx  AO label                                  w1            w2         delta
------------------------------------------------------------------------------
   1  0 N 2s                            0.20352698    0.00180617   -2.0172e-01
  15  1 N 2s                            0.20352698    0.00180617   -2.0172e-01
   5  0 N 2pz                           0.31430927    0.50608325    1.9177e-01
  19  1 N 2pz                           0.31430927    0.50608250    1.9177e-01
   2  0 N 3s                            0.99105364    0.99926762    8.2140e-03
  16  1 N 3s                            0.99105364    0.99926762    8.2140e-03
  20  1 N 3px                           0.99999324    0.99478537   -5.2079e-03
  21  1 N 3py                           0.99999324    0.99478537   -5.2079e-03
   7  0 N 3py                           0.99999324    0.99478538   -5.2079e-03
   6  0 N 3px                           0.99999324    0.99478538   -5.2079e-03
  25  1 N 3dz^2        

In [72]:
import numpy as np

def identify_occ_vir_redistribution(labels,
                                    w1_occ, w2_occ,
                                    w1_vir, w2_vir,
                                    *,
                                    min_weight=0.0,
                                    min_abs_delta=0.0,
                                    topn=None,
                                    print_table=True):
    """
    Identify AO directions showing occupied <-> virtual redistribution.

    Parameters
    ----------
    labels : list[str]
    w1_occ, w2_occ : array_like
        Occupied-space AO weights at geometry 1 and 2.
    w1_vir, w2_vir : array_like
        Virtual-space AO weights at geometry 1 and 2.
    min_weight : float
        Keep AO if any of occ/vir weights exceeds this threshold in either geometry.
    min_abs_delta : float
        Keep AO if max(|delta_occ|, |delta_vir|) >= min_abs_delta.
    topn : int or None
    print_table : bool

    Returns
    -------
    results : list of dict
    """
    labels = list(labels)
    w1_occ = np.asarray(w1_occ, dtype=float)
    w2_occ = np.asarray(w2_occ, dtype=float)
    w1_vir = np.asarray(w1_vir, dtype=float)
    w2_vir = np.asarray(w2_vir, dtype=float)

    n = len(labels)
    if not all(len(x) == n for x in [w1_occ, w2_occ, w1_vir, w2_vir]):
        raise ValueError("All input arrays must have the same length")

    d_occ = w2_occ - w1_occ
    d_vir = w2_vir - w1_vir

    # score = strongest redistribution magnitude
    score = np.maximum(np.abs(d_occ), np.abs(d_vir))

    keep = (
        (np.maximum.reduce([w1_occ, w2_occ, w1_vir, w2_vir]) > min_weight) &
        (score >= min_abs_delta)
    )
    idx = np.where(keep)[0]
    order = idx[np.argsort(score[idx])[::-1]]

    if topn is not None:
        order = order[:topn]

    results = []
    for i in order:
        # heuristic classification
        if d_occ[i] < 0 and d_vir[i] > 0:
            kind = "occ -> vir"
        elif d_occ[i] > 0 and d_vir[i] < 0:
            kind = "vir -> occ"
        else:
            kind = "mixed"

        results.append({
            "idx": int(i),
            "label": labels[i],
            "w1_occ": float(w1_occ[i]),
            "w2_occ": float(w2_occ[i]),
            "d_occ": float(d_occ[i]),
            "w1_vir": float(w1_vir[i]),
            "w2_vir": float(w2_vir[i]),
            "d_vir": float(d_vir[i]),
            "score": float(score[i]),
            "kind": kind,
        })

    if print_table:
        print(f"Filters: min_weight = {min_weight}, min_abs_delta = {min_abs_delta}")
        print(f"{'idx':>4}  {'AO label':<30}  {'d_occ':>12}  {'d_vir':>12}  {'score':>12}  {'type':>12}")
        print("-" * 92)
        for r in results:
            print(
                f"{r['idx']:4d}  "
                f"{r['label']:<30}  "
                f"{r['d_occ']:12.4e}  "
                f"{r['d_vir']:12.4e}  "
                f"{r['score']:12.4e}  "
                f"{r['kind']:>12}"
            )

    return results

In [73]:
redistributed = identify_occ_vir_redistribution(
    labels=labelsocc,
    w1_occ=w1occ, w2_occ=w2occ,
    w1_vir=w1vir, w2_vir=w2vir,
    min_weight=0.1,
    min_abs_delta=0.01,
    topn=20,
    print_table=True
)

Filters: min_weight = 0.1, min_abs_delta = 0.01
 idx  AO label                               d_occ         d_vir         score          type
--------------------------------------------------------------------------------------------
   1  0 N 2s                            2.0172e-01   -2.0172e-01    2.0172e-01    vir -> occ
  15  1 N 2s                            2.0172e-01   -2.0172e-01    2.0172e-01    vir -> occ
   5  0 N 2pz                          -1.9177e-01    1.9177e-01    1.9177e-01    occ -> vir
  19  1 N 2pz                          -1.9177e-01    1.9177e-01    1.9177e-01    occ -> vir
